# Sketch2Simulation — run the pipeline

Process sketch ➜ executable Aspen HYSYS script, through the nine-step multi-agent workflow.

**Before running**
1. `pip install -r requirements.txt` (or `pip install -e .`) in the kernel's environment.
2. Put your key in the environment or in a `.env` file at the repository root: `OPENAI_API_KEY=sk-...`
3. Start the kernel from the repository root (the notebook lives there), so `images/` and `sketch2sim/` resolve.
4. Step 9 (execution) needs Aspen HYSYS with COM on this Windows machine. Set `execute=False` to stop after the merged script is written.

Model names come from `sketch2sim/config.py` (override with `S2S_*` environment variables or edit the `PipelineConfig` below).


In [ ]:
import os

from sketch2sim import config
from sketch2sim.pipeline import PipelineConfig, run_pipeline

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is not set (export it or add it to .env).")

cfg = PipelineConfig(
    image_path="images/cs1.png",   # cs1 desalting | cs2 Merox | cs3 crude distillation | cs4 aromatics
    log_dir="logs",
    output_dir="outputs",
    # --- models (defaults from sketch2sim/config.py; override via S2S_* env vars or here) ---
    descriptor_model=config.DESCRIPTOR_MODEL,
    description_validator_model=config.DESCRIPTION_VALIDATOR_MODEL,
    extractor_model=config.EXTRACTOR_MODEL,
    basis_model=config.BASIS_MODEL,
    instantiation_model=config.INSTANTIATION_MODEL,
    configuration_model=config.CONFIGURATION_MODEL,
    fixer_model=config.FIXER_MODEL,
    # --- instruction files for the two coding agents ---
    instantiation_instruction_files=["sketch2sim/instructions/instantiation_instructions_simple_v5.txt"],
    configuration_instruction_files=["sketch2sim/instructions/configuration_instructions_simple_v5.txt"],
    # --- behaviour ---
    max_runtime_fix_retries=3,
    extractor_uses_image=True,   # paper: the extractor sees the sketch as well as the description
    execute=True,                # False -> stop after merging (no HYSYS needed)
)
cfg.models()


In [ ]:
result = run_pipeline(cfg)

print("\nDescription validation:", result.validation.get("status"))
print("Basis script:        ", result.basis_path)
print("Instantiation script:", result.instantiation_path)
print("Configuration script:", result.configuration_path)
print("Merged script:       ", result.combined_path)
print("Executed in HYSYS:   ", result.executed)


In [ ]:
# Inspect the generated HYSYS automation script
from pathlib import Path

print(Path(result.combined_path).read_text(encoding="utf-8"))
